In [ ]:
import os
os.environ['CNV_NOTEBOOK_RFB'] = '1'   # in-kernel render, backpressure interaction

import compneurovis as cnv

# IMPORTANT: build() runs in the CHILD process. Construct everything from
# scratch and return the configured source. Capture no live NEURON objects in
# the closure (cloudpickle ships the function, not live h.Section objects).
def build():
    import pathlib
    from neuron import h
    import compneurovis as cnv
    from compneurovis.backends.neuron.utils import load_swc_neuron

    repo = pathlib.Path(cnv.__file__).resolve().parents[2]
    secs = load_swc_neuron(str(repo / 'res' / 'Animal_2_Basal_2.CNG.swc'))
    for s in secs:
        s.insert('hh')
        if 'soma' not in s.name().lower():
            s.nseg = 10

    soma = next(s for s in secs if 'soma' in s.name().lower())
    stim = h.IClamp(soma(0.5)); stim.delay = 0.0; stim.dur = 1e9; stim.amp = 0.5
    h.dt = 0.025; h.celsius = 6.3; h.finitialize(-65.0)

    sim = cnv.source(cnv.neuron.attach(sections=secs))
    sim.morphology()
    sim.control(
        'clamp_amp',
        label='IClamp amplitude (nA)',
        get=lambda: float(stim.amp),
        set=lambda v: setattr(stim, 'amp', float(v)),
        min=-0.5,
        max=2.0,
    )
    return sim

widget = cnv.show(build)   # sim runs in a child process; render stays in-kernel
widget